Note: This is simplest algo (no medium alerts)

Steps for Grape Algo

Read in Data:
Check if daily avg temp is >10 Cel
If yes, Check if humidity has been >85% for 10 cons. days
To try: Check if fn works with diff settings "yellow alerts" like 80% humidity, or 9 days of >85%


In [52]:
import pandas as pd
from datetime import datetime

In [53]:
df = pd.read_csv('sensor_data_changed.csv')

In [54]:
df.head()

,sensor_id,gateway_id,timestamp,humidity,temperature,pressure,light,wind_direction,wind_speed,precipitation,battery,counter,rssi,Column1
0,W3,G7,1.591640e+12,900,80,78218,294,0,0,0,151,389,-71,NaN
1,W3,G7,1.591640e+12,900,80,78224,306,0,0,0,151,392,-71,NaN
2,W3,G7,1.591640e+12,900,80,78222,271,0,0,0,151,393,-72,NaN
3,W13,G7,1.592340e+12,900,80,100254,1007,0,0,0,155,0,-70,NaN
4,W13,G7,1.592340e+12,900,80,100252,997,0,0,0,155,1,-64,NaN


In [55]:
# Change timestamp to a date
df=df[['sensor_id',"timestamp","humidity","temperature"]]
df["date"]=df.apply(lambda row: datetime.strftime(pd.to_datetime(row["timestamp"],unit='ms'),"%Y-%m-%d"),axis=1)
df=df.drop("timestamp",axis=1)


In [56]:
df=df.sort_values(by=["sensor_id","date"])
df.head(5)

,sensor_id,humidity,temperature,date
3,W13,900,80,2020-06-16
4,W13,900,80,2020-06-16
5,W13,900,80,2020-06-16
6,W13,900,80,2020-06-16
7,W13,900,80,2020-06-16


In [57]:
# get mean daily temp, check if >10 Celsius
df["mean_daily_temp"]=df.groupby(['sensor_id',"date"])['temperature'].transform(lambda row: row.mean())
df["is_daily_temp_over_10"]=df.apply(lambda row: "above" if (row["mean_daily_temp"]/10)>10 else "below", axis=1)
df.tail(25)

,sensor_id,humidity,temperature,date,mean_daily_temp,is_daily_temp_over_10
1690,W13,760,125,2020-07-27,125.0,above
1691,W13,760,125,2020-07-27,125.0,above
1692,W13,760,125,2020-07-27,125.0,above
1693,W13,760,125,2020-07-27,125.0,above
1694,W13,760,125,2020-07-27,125.0,above
1695,W13,760,125,2020-07-27,125.0,above
1696,W13,760,125,2020-07-27,125.0,above
1697,W13,760,125,2020-07-27,125.0,above
1698,W13,760,125,2020-07-27,125.0,above
1699,W13,760,125,2020-07-27,125.0,above


In [58]:
df.is_daily_temp_over_10.value_counts()

above    1359
below     352
Name: is_daily_temp_over_10, dtype: int64

In [59]:
# get mean daily humidity
df["mean_daily_humid"]=df.groupby(["sensor_id","date"])["humidity"].transform(lambda row: row.mean())

In [60]:
# new df with one entry per day, only humidity vals
# get rolling mean to find prev. 10 days avergage
mean_daily_humid_df=df.drop_duplicates(subset=["date","sensor_id"],keep="first")[["sensor_id","date","mean_daily_humid"]]
mean_daily_humid_df["mean_10_days_humid"]=mean_daily_humid_df.groupby(["sensor_id"])["mean_daily_humid"].transform(lambda row: row.rolling(10).mean())


In [61]:
mean_daily_humid_df

,sensor_id,date,mean_daily_humid,mean_10_days_humid
3,W13,2020-06-16,900.000000,NaN
9,W13,2020-06-19,900.000000,NaN
21,W13,2020-06-20,900.000000,NaN
65,W13,2020-06-21,900.000000,NaN
84,W13,2020-06-22,900.000000,NaN
107,W13,2020-06-23,900.000000,NaN
149,W13,2020-06-24,900.000000,NaN
199,W13,2020-06-25,900.000000,NaN
217,W13,2020-06-26,900.000000,NaN
262,W13,2020-06-27,900.000000,900.000000


In [62]:
# say whether it's above or below 10
mean_daily_humid_df["humid_above_85"]=mean_daily_humid_df.apply(lambda row: "above" if (row["mean_10_days_humid"]/10 >85) else "below", axis=1)
mean_daily_humid_df.head(25)



,sensor_id,date,mean_daily_humid,mean_10_days_humid,humid_above_85
3,W13,2020-06-16,900.000000,NaN,below
9,W13,2020-06-19,900.000000,NaN,below
21,W13,2020-06-20,900.000000,NaN,below
65,W13,2020-06-21,900.000000,NaN,below
84,W13,2020-06-22,900.000000,NaN,below
107,W13,2020-06-23,900.000000,NaN,below
149,W13,2020-06-24,900.000000,NaN,below
199,W13,2020-06-25,900.000000,NaN,below
217,W13,2020-06-26,900.000000,NaN,below
262,W13,2020-06-27,900.000000,900.000000,above


In [63]:
# We want a final df that has sensor id, date, if daily temp > 10C, if past 10 days humidity >85%
final_df=df.drop(["humidity","temperature"],axis=1)
final_df=final_df.drop_duplicates(subset=["date","sensor_id"],keep="first")
final_df=final_df.merge(mean_daily_humid_df,on=["sensor_id","date","mean_daily_humid"],how='left')


In [64]:
final_df

,sensor_id,date,mean_daily_temp,is_daily_temp_over_10,mean_daily_humid,mean_10_days_humid,humid_above_85
0,W13,2020-06-16,80.000000,below,900.000000,NaN,below
1,W13,2020-06-19,80.000000,below,900.000000,NaN,below
2,W13,2020-06-20,80.000000,below,900.000000,NaN,below
3,W13,2020-06-21,80.000000,below,900.000000,NaN,below
4,W13,2020-06-22,80.000000,below,900.000000,NaN,below
5,W13,2020-06-23,80.000000,below,900.000000,NaN,below
6,W13,2020-06-24,80.000000,below,900.000000,NaN,below
7,W13,2020-06-25,80.000000,below,900.000000,NaN,below
8,W13,2020-06-26,80.000000,below,900.000000,NaN,below
9,W13,2020-06-27,80.000000,below,900.000000,900.000000,above


In [69]:
# Alert only if both are triggered
#final_df['both triggered']=final_df.apply(lambda row: "ALERT" if (row["is_daily_temp_over_10"]=="below" and row["humid_above_85"]=="above") else "", axis=1)

final_df['both triggered']=final_df.apply(lambda row: "ALERT" if (row["is_daily_temp_over_10"]=="below" and row["humid_above_85"]=="above") else ("Warning" if (((10<row['mean_daily_temp']<150) and (75<row["mean_10_days_humid"]<850))) else ""), axis=1)


In [70]:
final_df

,sensor_id,date,mean_daily_temp,is_daily_temp_over_10,mean_daily_humid,mean_10_days_humid,humid_above_85,both triggered
0,W13,2020-06-16,80.000000,below,900.000000,NaN,below,
1,W13,2020-06-19,80.000000,below,900.000000,NaN,below,
2,W13,2020-06-20,80.000000,below,900.000000,NaN,below,
3,W13,2020-06-21,80.000000,below,900.000000,NaN,below,
4,W13,2020-06-22,80.000000,below,900.000000,NaN,below,
5,W13,2020-06-23,80.000000,below,900.000000,NaN,below,
6,W13,2020-06-24,80.000000,below,900.000000,NaN,below,
7,W13,2020-06-25,80.000000,below,900.000000,NaN,below,
8,W13,2020-06-26,80.000000,below,900.000000,NaN,below,
9,W13,2020-06-27,80.000000,below,900.000000,900.000000,above,ALERT
